# Estimación de peso de pollos

## Colegio de Posgraduados

### COA661 Inteligencia Artificial

Profesor: Dr. Juan Manuel González Camacho

Entrega: José Alfredo Martínez

En este notebook se entrenan 4 modelos para predecir el peso de los pollos: Support Vector Regressor, Random Forest Regressor, K Nearest Neighbors y Multilayer Perceptron.

Se utilizan 2 Componentes Principales del Dataset

In [1]:
# Librerias
import numpy as np
import pandas as pd

import optuna

from IPython.display import clear_output

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import sklearn.model_selection

import joblib

## Código

In [2]:
# Cargar datos
datos1 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_train')
datos2 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_test')

fil, col = datos1.shape

X_train = datos1.iloc[:, 0 : col - 1].to_numpy()
Y_train = datos1.iloc[:, col - 1].to_numpy()

X_test = datos2.iloc[:, 0 : col - 1].to_numpy()
Y_test = datos2.iloc[:, col - 1].to_numpy()

In [3]:
# Cargar datos de estandarización y PCA
scx = joblib.load('T_scaler8.pkl')
pca = joblib.load('T_PCA2.pkl')

In [4]:
# Estandarizar los datos
X_train_std = scx.transform(X_train)
X_test_std = scx.transform(X_test)

X_train_pca = pca.transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

print(X_train_std.shape)
print(X_train_pca.shape)

(2439, 8)
(2439, 2)


## Support Vector Regressor

In [5]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    kernel = trial.suggest_categorical('kernel', ['rbf'])
    C = trial.suggest_float('C', 0.1, 100, log = True)
    gamma = trial.suggest_float('gamma', 0.001, 10, log = True)
    epsilon = trial.suggest_float('epsilon', 0.0001, 1)

    # Crear el modelo y realizar la Cross Validation
    modelo = SVR(kernel = kernel, C = C, gamma = gamma, epsilon = epsilon)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_pca, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:18:24,812] Trial 199 finished with value: 0.49865331377418354 and parameters: {'kernel': 'rbf', 'C': 1.852983817082758, 'gamma': 8.061588085269147, 'epsilon': 0.08700083102336456}. Best is trial 122 with value: 0.5088948154019602.


Cross Validation
Mejores hiperparámetros:  {'kernel': 'rbf', 'C': 1.6440364132322474, 'gamma': 9.423243821425594, 'epsilon': 0.11195575136996981}
R2:  0.51


In [6]:
# Guardar datos de modelo
svr = SVR(**study.best_params)
joblib.dump(svr, './modelo_svr_pca2.pkl')

['./modelo_svr_pca2.pkl']

## Random Forest Regressor

In [7]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_depth = trial.suggest_int('max_depth', 2, 35)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    modelo = RandomForestRegressor(n_estimators = n_estimators, max_depth = max_depth, min_samples_split = min_samples_split)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_pca, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:22:33,917] Trial 199 finished with value: 0.6012458798642698 and parameters: {'n_estimators': 440, 'max_depth': 29, 'min_samples_split': 3}. Best is trial 107 with value: 0.6059933476629917.


Cross Validation
Mejores hiperparámetros:  {'n_estimators': 428, 'max_depth': 23, 'min_samples_split': 2}
R2:  0.61


In [8]:
# Guardar datos de modelo
rf = RandomForestRegressor(**study.best_params)
joblib.dump(rf, './modelo_rf_pca2.pkl')

['./modelo_rf_pca2.pkl']

## K Neighbors Regressor

In [9]:
def objective(trial):
    clear_output(wait=True)
    
    n_neighbors = trial.suggest_int('n_neighbors', 1, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan'])

    modelo = KNeighborsRegressor(n_neighbors = n_neighbors, weights = weights, metric = metric)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_pca, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:22:37,431] Trial 199 finished with value: 0.6370243701306704 and parameters: {'n_neighbors': 42, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 30 with value: 0.6486940851132383.


Cross Validation
Mejores hiperparámetros:  {'n_neighbors': 19, 'weights': 'distance', 'metric': 'euclidean'}
R2:  0.65


In [10]:
# Guardar datos de modelo
knn = KNeighborsRegressor(**study.best_params)
joblib.dump(knn, './modelo_knn_pca2.pkl')

['./modelo_knn_pca2.pkl']

## Multilayer Perceptron Regressor

In [11]:
def objective(trial):
    clear_output(wait=True)

    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(25,), (50,), (25, 25), (50, 25)])
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    alpha = trial.suggest_float('alpha', 0.00001, 0.1, log = True)
    learning_rate_init = trial.suggest_float('learning_rate_init', 0.0001, 0.1, log = True)

    modelo = MLPRegressor(hidden_layer_sizes = hidden_layer_sizes, activation = activation, alpha = alpha, learning_rate_init = learning_rate_init, max_iter = 2000, random_state = 42)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_pca, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()

    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:23:34,060] Trial 199 finished with value: 0.443108275371402 and parameters: {'hidden_layer_sizes': (50, 25), 'activation': 'tanh', 'alpha': 0.003210340049265388, 'learning_rate_init': 0.0049859351988319}. Best is trial 111 with value: 0.48802591343324886.


Cross Validation
Mejores hiperparámetros:  {'hidden_layer_sizes': (50, 25), 'activation': 'relu', 'alpha': 0.003181092312540986, 'learning_rate_init': 0.00432968969886045}
R2:  0.49


In [12]:
# Guardar datos de modelo
mlp = MLPRegressor(**study.best_params)
joblib.dump(mlp, './modelo_mlp_pca2.pkl')

['./modelo_mlp_pca2.pkl']